# DataBay Quality Examples

Examples for `regex_check` and `row_level_rules`.

In [14]:
from pyspark.sql import SparkSession, Row
from databay import regex_check, row_level_rules, find_key_set

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

df_quality = spark.createDataFrame([
    Row(id=1, email="alice@example.com", amount=100.0, country="US"),
    Row(id=2, email="bad-email", amount=-5.0, country="SK"),
    Row(id=3, email="bob@example.org", amount=0.0, country="USA"),
])

In [2]:
data = [
    ("john.doe@gmail.com",   "+421912345678", "81101", "850101/1234"),
    ("alice@test.sk",        "+421903456789", "04001", "991231/0001"),
    ("bob@yahoo.com",        "+421915555555", "91701", "750505/4321"),
    ("badmail",              "+421900000000", "ABCDE", "8501011234"),
    ("noatsign.sk",          "0912345678",    "1234",  "abc"),
    ("maria@gmail",          "+42191234567",  "99999", None),
    ("peter@firma.sk",       None,            "82109", "700101/1111"),
    (None,                   "+421912345678", "01001", "850101/1234"),
    ("eva@domain.com",       "+421944444444", "81101", "123456/7890"),
    ("test@",                "+421933333333", "00000", "990101/0000"),

    ("valid1@x.sk",          "+421911111111", "90001", "880101/2222"),
    ("valid2@x.sk",          "+421922222222", "90002", "880102/2223"),
    ("valid3@x.sk",          "+421933333333", "90003", "880103/2224"),
    ("valid4@x.sk",          "+421944444444", "90004", "880104/2225"),
    ("valid5@x.sk",          "+421955555555", "90005", "880105/2226"),

    ("x@x.x",                "+421966666666", "12A45", "101010/1010"),
    ("toolongemailaddress@test.sk", "+421977777777", "54321", "1010101010"),
    ("wrong@@test.sk",       "+421988888888", "ABCDE", "abcd"),
    ("correct@email.com",    "+421999999999", "11111", "650101/3333"),
    ("abc@abc.sk",           "+421900000000", "22222", None),

    ("mail@test.com",        "123456789",     "33333", "990229/1234"),
    ("mail2@test.com",       "+42191234",     "44444", "850101/12"),
    ("mail3@test.com",       "+421912345678", "55555", "850101/1234"),
    ("mail4@test.com",       "+421912345678", "66666", "850101/1234"),
    ("mail5@test.com",       "+421912345678", "77777", "850101/1234"),

    ("brokenmail",           None,            None,    None),
    ("another@test.sk",      "+421912345678", "88888", "010101/0001"),
    ("another@test.sk",      "+421912345678", "88888", "010101/0001"),
    ("dup@test.sk",          "+421912345678", "99999", "010101/0001"),
    ("dup@test.sk",          "+421912345678", "99999", "010101/0001"),
]

df = spark.createDataFrame(
    data,
    ["email", "phone", "zip_code", "rodne_cislo"]
)

In [3]:
rules = {
    "email": r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    "phone": r"^\+4219\d{8}$",
    "zip_code": r"^\d{5}$",
    "rodne_cislo": r"^\d{6}/\d{4}$"
}

regex_check(df, rules, show_summary_only=True).show(truncate=False)

+-----------+------------------------------------------------+----------+-------------+-----------------+----------------+
|column_name|pattern                                         |total_rows|matching_rows|non_matching_rows|match_percentage|
+-----------+------------------------------------------------+----------+-------------+-----------------+----------------+
|email      |^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$|30        |22           |8                |73.3333         |
|phone      |^\+4219\d{8}$                                   |30        |24           |6                |80.0            |
|zip_code   |^\d{5}$                                         |30        |25           |5                |83.3333         |
|rodne_cislo|^\d{6}/\d{4}$                                   |30        |22           |8                |73.3333         |
+-----------+------------------------------------------------+----------+-------------+-----------------+----------------+



In [4]:
rules = {
    "phone": r"^\+4219\d{8}$"
}

regex_check(df, rules, show_summary_only=False, top_n=5).show(truncate=False)

+-----------+-------------+------------------+------------------+
|column_name|pattern      |non_matching_value|non_matching_count|
+-----------+-------------+------------------+------------------+
|phone      |^\+4219\d{8}$|<NULL>            |2                 |
|phone      |^\+4219\d{8}$|0912345678        |1                 |
|phone      |^\+4219\d{8}$|+42191234567      |1                 |
|phone      |^\+4219\d{8}$|123456789         |1                 |
|phone      |^\+4219\d{8}$|+42191234         |1                 |
+-----------+-------------+------------------+------------------+



In [5]:
rules = {
    "rodne_cislo": r"^\d{6}/\d{4}$"
}

regex_check(df, rules, show_summary_only=False, top_n=5).show(truncate=False)

+-----------+-------------+------------------+------------------+
|column_name|pattern      |non_matching_value|non_matching_count|
+-----------+-------------+------------------+------------------+
|rodne_cislo|^\d{6}/\d{4}$|<NULL>            |3                 |
|rodne_cislo|^\d{6}/\d{4}$|8501011234        |1                 |
|rodne_cislo|^\d{6}/\d{4}$|abc               |1                 |
|rodne_cislo|^\d{6}/\d{4}$|1010101010        |1                 |
|rodne_cislo|^\d{6}/\d{4}$|abcd              |1                 |
+-----------+-------------+------------------+------------------+



In [6]:
data_ico = [
    ("12345678",),
    ("87654321",),
    ("00000000",),
    ("1234567",),
    ("123456789",),
    ("ABC12345",),
    ("12 345678",),
    ("99999999",),
    ("11111111",),
    ("22222222",),
    ("33333333",),
    ("44444444",),
    ("55555555",),
    ("66666666",),
    ("77777777",),
    ("88888888",),
    ("12-345678",),
    ("",),
    (None,),
    ("00001234",),
    ("12039847",),
    ("56000001",),
    ("00112233",),
    ("12A45678",),
    ("8765 4321",),
    ("1234abcd",),
    ("9999999",),
    ("abcdefgh",),
    ("12345670",),
    ("76543210",),
]

df_ico = spark.createDataFrame(data_ico, ["ico"])

rules = {
    "ico": r"^\d{8}$"
}

regex_check(df_ico, rules, show_summary_only=True).show(truncate=False)

+-----------+-------+----------+-------------+-----------------+----------------+
|column_name|pattern|total_rows|matching_rows|non_matching_rows|match_percentage|
+-----------+-------+----------+-------------+-----------------+----------------+
|ico        |^\d{8}$|30        |18           |12               |60.0            |
+-----------+-------+----------+-------------+-----------------+----------------+



In [7]:
data_eic = [
    ("24Z1234567890123",),
    ("24ZABCDEFGHIJKLM",),
    ("24ZABCDEF1234567",),
    ("24Z0000000000000",),
    ("24Z9999999999999",),
    ("24ZABC123ABC1234",),
    ("24Z123",),
    ("24Z12345678901234",),
    ("23Z1234567890123",),
    ("24z1234567890123",),
    ("24Z123456789012A",),
    ("24ZABCDEFGHIJKL1",),
    ("24ZABCDEFGHIJKL@",),
    ("24ZABCDEFGHIJKL",),
    ("",),
    (None,),
    ("24ZABCABCABCABC1",),
    ("24Z1111111111111",),
    ("24Z2222222222222",),
    ("24Z3333333333333",),
    ("24Z4444444444444",),
    ("24Z5555555555555",),
    ("24Z6666666666666",),
    ("24Z7777777777777",),
    ("24Z8888888888888",),
    ("24Z999999999999A",),
    ("24ZABCDEF12345",),
    ("24ZABCDEF12345678",),
    ("ZZZ1234567890123",),
    ("24ZABCDEABCDEAB1",),
]

df_eic = spark.createDataFrame(data_eic, ["eic"])

rules = {
    "eic": r"^24Z[A-Z0-9]{13}$"
}

regex_check(df_eic, rules, show_summary_only=True).show(truncate=False)

+-----------+-----------------+----------+-------------+-----------------+----------------+
|column_name|pattern          |total_rows|matching_rows|non_matching_rows|match_percentage|
+-----------+-----------------+----------+-------------+-----------------+----------------+
|eic        |^24Z[A-Z0-9]{13}$|30        |19           |11               |63.3333         |
+-----------+-----------------+----------+-------------+-----------------+----------------+



In [8]:
keys_data = [
    ("EIC001",),
    ("EIC002",),
    ("EIC003",),
    ("EIC004",),
    ("EIC005",),
    ("EIC006",),
    ("EIC007",),
    ("EIC008",),
    ("EIC009",),
    ("EIC010",),
]

keys_df = spark.createDataFrame(keys_data, ["eic"])

In [9]:
contracts_data = [
    ("EIC001", "C1"),
    ("EIC002", "C2"),
    ("EIC003", "C3"),
    ("EIC004", "C4"),
    ("EIC005", "C5"),
    ("EIC006", "C6"),
    ("EIC007", "C7"),
    ("EIC008", "C8"),
    ("EIC009", "C9"),
    ("EIC999", "C999"),  # cudzí kľúč
]

contracts_df = spark.createDataFrame(
    contracts_data,
    ["eic_code", "contract_id"]
)

In [10]:
metering_data = [
    ("M1", "EIC001"),
    ("M2", "EIC002"),
    ("M3", "EIC003"),
    ("M4", "EIC010"),
    ("M5", "EIC777"),  # mimo zoznamu
]

metering_df = spark.createDataFrame(
    metering_data,
    ["meter_id", "connection_eic"]
)

In [11]:
billing_data = [
    ("INV1", "EIC900"),
    ("INV2", "EIC901"),
]

billing_df = spark.createDataFrame(
    billing_data,
    ["invoice_id", "eic"]
)

In [12]:
network_data = [
    ("EIC001",),
    ("EIC001",),
    ("EIC005",),
    ("EIC006",),
    ("EIC010",),
    (None,),
]

network_df = spark.createDataFrame(
    network_data,
    ["grid_eic"]
)

In [15]:
tables = {
    "contracts": contracts_df,
    "metering": metering_df,
    "billing": billing_df,
    "network": network_df,
}

result = find_key_set(
    tables=tables,
    keys_df=keys_df,
    key_column="eic",
    candidate_columns=None,
    top_n=3,
)

result.show(truncate=False)

+----------+--------------+----------+-------------+-------------+------------+----------------------+
|table_name|column_name   |total_keys|matched_count|missing_count|coverage_pct|missing_keys_sample   |
+----------+--------------+----------+-------------+-------------+------------+----------------------+
|contracts |eic_code      |10        |9            |1            |90.0        |EIC010                |
|contracts |contract_id   |10        |0            |10           |0.0         |EIC001, EIC002, EIC003|
|metering  |meter_id      |10        |0            |10           |0.0         |EIC001, EIC002, EIC003|
|metering  |connection_eic|10        |4            |6            |40.0        |EIC004, EIC005, EIC006|
|billing   |invoice_id    |10        |0            |10           |0.0         |EIC001, EIC002, EIC003|
|billing   |eic           |10        |0            |10           |0.0         |EIC001, EIC002, EIC003|
|network   |grid_eic      |10        |4            |6            |40.0   